# 07 · Models

## Setup

There is one `.env` for the whole monorepo, at the repo root. From `notebooks/module-0/`
that is two levels up. See [notebooks/README.md](../README.md).

In [1]:
import os

from dotenv import load_dotenv

load_dotenv("../../.env")

HAS_GROQ = bool(os.environ.get("GROQ_API_KEY"))
HAS_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))

## What is a chat model?

A chat model takes a list of messages and returns a message.

[`init_chat_model`](https://docs.langchain.com/oss/python/langchain/models) creates one
from a `"{provider}:{model}"` string, so switching providers is a string edit rather
than an import change.

In [2]:
from typing import Annotated

from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph, add_messages
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

In [3]:
GROQ_MODEL = "groq:openai/gpt-oss-20b"
OPENAI_MODEL = "openai:gpt-4o-mini"

groq = init_chat_model(GROQ_MODEL, temperature=0)
openai = init_chat_model(OPENAI_MODEL, temperature=0)

Model names change often. `groq/openai-gpt-oss-20b` and `gpt-4o-mini` are what this
repo's keys could reach when the notebook was written; if a cell 404s, list what your
account actually offers rather than guessing.

The provider classes work too, and are what `init_chat_model` builds underneath:

```python
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

groq = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
openai = ChatOpenAI(model="gpt-4o-mini", temperature=0)
```

Use the classes when you need a provider-specific argument; use `init_chat_model` for
everything else.

## Invoke

Pass a string, or a list of messages for more control.

In [4]:
groq_response = groq.invoke("Good morning, How are you doing?")

In [5]:
groq_response

AIMessage(content='Good morning! I’m doing great—thanks for asking. How about you? Anything exciting on your agenda today?', additional_kwargs={'reasoning_content': 'User says "Good morning, How are you doing?" It\'s a greeting. We should respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 79, 'total_tokens': 132, 'completion_time': 0.056688237, 'completion_tokens_details': {'reasoning_tokens': 21}, 'prompt_time': 0.003821216, 'prompt_tokens_details': None, 'queue_time': 0.192157822, 'total_time': 0.060509453}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c9afb2bdb4', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0897c-670d-7241-b376-69ac20aed711-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 79, 'output_tokens': 53, 'total_tokens': 132, 'output_token_details': {'reasoning': 21}})

The return value is an `AIMessage` — the same message type you hand-built in 06.
Alongside `.content` it carries usage metadata:

## Parameters

`temperature` controls randomness, `max_tokens` caps the response length. Both are
set at construction, or per call with `.bind(...)`.

In [6]:
creative = init_chat_model(
    OPENAI_MODEL,
    temperature=1.0,
    max_tokens=64,
)
print(creative.invoke("What is the capital city of mars?").content)

Mars does not have a capital city, as it is a planet that is currently not inhabited by humans. Although there are discussions and plans for potential human settlements on Mars in the future, there is no established government or city structure on the planet at this time.


## Stream

`.stream()` yields chunks as they arrive, which is what makes a UI feel responsive.

In [7]:
for i, chunk in enumerate(openai.stream("Count from 1 to 5, separated by commas.")):
    if chunk.content:
        print(chunk.content, end="", flush=True)
    if i > 200:            # reasoning models can emit a lot of small chunks
        break
print()

1, 2, 3, 4, 5


## Batch

`.batch()` sends independent prompts in parallel — one round trip's worth of latency
instead of several.

In [8]:

answers = groq.batch([
    "Say the word: alpha",
    "Say the word: beta",
    "Say the word: gamma",
])
for answer in answers:
    print(repr(answer.content))

'alpha'
'beta'
'gamma'
